# S5.M1-2 方法筛选：方法 × 数据集的增益矩阵与组合

读入 S5.M1 的 14 个任务结果，输出公开汇总：各方法测试 AUROC、全部配对对比（区间、p、Holm p）、每个对比在多少个数据集上有增益，以及选定组合的计算量。

**口径**（[DR-M5-012](../../../registry/decision_records/DR-M5-012.yaml) · D-4）：6 个 seed，点估计为均值；95% 区间来自两层 bootstrap（测试客户簇 × seed，1,000 次）；每个数据集全部对比为一族做 Holm。「增益」= 区间不含 0 且为正；Holm p ≥ 0.05 的写作「单数据集名义显著」。开发证据，不代表汇丰效果。

In [1]:
import hashlib
import json
import os
import pathlib
import subprocess
import sys
import tempfile

import numpy as np
import pandas as pd
import yaml

HASH_PREFIX_LEN = 12
REPO_ROOT = pathlib.Path.cwd()
while not (REPO_ROOT / "AGENTS.md").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))
CONFIG_PATH = REPO_ROOT / "modules/m5_modeling/configs/s5_m1_method_screen.yaml"
cfg = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
COMPONENTS = [REPO_ROOT / "modules/m5_modeling/components/method_screen.py",
              REPO_ROOT / "modules/m5_modeling/components/method_screen_data.py"]


def sha(path, n=HASH_PREFIX_LEN):
    return hashlib.sha256(pathlib.Path(path).read_bytes()).hexdigest()[:n]


git_sha = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True, cwd=REPO_ROOT).strip()
dirty = subprocess.run(["git", "status", "--porcelain"], capture_output=True, text=True,
                       cwd=REPO_ROOT).stdout.strip() != ""
print("config:", CONFIG_PATH.relative_to(REPO_ROOT), "sha256:" + sha(CONFIG_PATH))
print("决策记录:", cfg["decision_record"], "| step:", cfg["step_id"], "|", cfg["purpose"])
print("seed:", cfg["seed"], "| seeds:", cfg["seeds"])
print("git:", git_sha + ("（工作区含本步未提交改动）" if dirty else ""))
for c in COMPONENTS:
    print("组件:", c.relative_to(REPO_ROOT), "sha256:" + sha(c))
PRIV = pathlib.Path(os.environ[cfg["private_root_env"]])
RAW = pathlib.Path(os.environ[cfg["raw_root_env"]])
RUN = pathlib.Path(os.environ[cfg["run_dir_env"]])
NAMES = cfg["datasets"]
R = {n: json.loads((RUN / (n + ".json")).read_text(encoding="utf-8")) for n in NAMES}
print("运行结果：%d / %d 个数据集" % (len(R), len(NAMES)))
assert set(R) == set(NAMES)

config: modules/m5_modeling/configs/s5_m1_method_screen.yaml sha256:761bae66da90
决策记录: registry/decision_records/DR-M5-012.yaml | step: S5.M1 | 文献优化方法（I2 / I3 / I6 / M1 / M2 / M4 / M5 / M6）在 14 个公开数据集任务上的增益、组合与缺口
seed: 11 | seeds: [11, 22, 33, 44, 55, 66]
git: 75c3bb8（工作区含本步未提交改动）
组件: modules/m5_modeling/components/method_screen.py sha256:05e19e025d0b
组件: modules/m5_modeling/components/method_screen_data.py sha256:e4bfdf6fe3d4
运行结果：14 / 14 个数据集


## 1. 各方法的测试 AUROC

In [2]:
OUT = REPO_ROOT / cfg["results_dir"]
OUT.mkdir(parents=True, exist_ok=True)
SHOW = ["G_t", "F_def", "F_bins", "F_tune", "F_opt", "F_opt_p", "F_M4", "F_M1", "G_x", "F_M1x", "F_M6", "M2_fuse",
        "M2_student", "M2_student_x", "DP_hard_e2", "DP_soft_e2", "DP_hard_e4", "DP_soft_e4", "COMBO"]
test = pd.DataFrame({n: {k: R[n]["test"][k]["auroc"] for k in SHOW if k in R[n]["test"]} for n in NAMES}).T
test["combo_choice"] = [R[n]["val"]["combo_choice"] for n in NAMES]
test.to_csv(OUT / "test_auroc.csv", float_format="%.5f")
print("测试 AUROC（6 seed 均值）：")
print(test.round(cfg["round_auc"]).to_string())

测试 AUROC（6 seed 均值）：
                     G_t   F_def  F_bins  F_tune   F_opt  F_opt_p    F_M4    F_M1    F_M6  M2_fuse  M2_student  DP_hard_e2  DP_soft_e2  DP_hard_e4  DP_soft_e4   COMBO     G_x   F_M1x  M2_student_x combo_choice
santander_g3      0.8264  0.8697  0.8699  0.8726  0.8722   0.8748  0.8748  0.8776  0.8753   0.8748      0.8269      0.8621      0.8589      0.8731      0.8724  0.8776     NaN     NaN           NaN           M1
mock_balanced     0.8277  0.8907  0.8968  0.8968  0.9015   0.8995  0.8995  0.8917  0.9060   0.9012      0.8348      0.8106      0.7866      0.8838      0.8661  0.9060     NaN     NaN           NaN         P+M6
mock_overlap20    0.8605  0.8928  0.8979  0.8928  0.9000   0.9009  0.8978  0.8870  0.9055   0.8994      0.8605      0.7962      0.8141      0.8718      0.8731  0.8978  0.8648  0.8933        0.8648         P_M4
mock_overlap50    0.8694  0.9055  0.9061  0.9034  0.9067   0.9061  0.9079  0.8997  0.9126   0.9066      0.8683      0.8157      0.8215     

## 2. 配对对比与按方法汇总

In [3]:
HOLM_ALPHA = 0.05
con = pd.DataFrame([{"dataset": n, **c} for n in NAMES for c in R[n]["contrasts"]])
con = con[["dataset", "label", "a", "b", "diff", "ci_low", "ci_high", "p", "p_holm"]]
con.to_csv(OUT / "contrasts.csv", index=False, float_format="%.5f")
con["结论"] = np.where(con["ci_low"] > 0, "增益", np.where(con["ci_high"] < 0, "变差", "不显著"))
summ = con.groupby("label", sort=False)["结论"].value_counts().unstack(fill_value=0)
summ["增益数据集"] = con[con["结论"] == "增益"].groupby("label", sort=False)["dataset"].agg(", ".join)
summ["其中 Holm p < 0.05"] = con[(con["结论"] == "增益") & (con["p_holm"] < HOLM_ALPHA)].groupby(
    "label", sort=False)["dataset"].agg(", ".join)
summ = summ.fillna("")
summ.to_csv(OUT / "method_summary.csv")
print("每个对比在多少个数据集上：增益（区间不含 0 且为正）/ 变差 / 不显著：")
print(summ.to_string())

每个对比在多少个数据集上：增益（区间不含 0 且为正）/ 变差 / 不显著：
结论                        不显著  变差  增益                                                                                                                                                         增益数据集                                                                                                                               其中 Holm p < 0.05
label                                                                                                                                                                                                                                                                                                                                             
I2 只细分桶                    10   1   3                                                                                                                    mock_balanced, orange_upsell, santander_tx                                                                                        

## 3. 选定组合的计算量

In [4]:
cost = pd.DataFrame([{"数据集": n, "选定组合": R[n]["val"]["combo_choice"],
                      "原版联邦": R[n]["costs"]["F_def"], "优化版联邦": R[n]["costs"]["F_opt_p"],
                      "选定组合计算量": R[n]["costs"]["COMBO"],
                      "组合 / 原版": R[n]["costs"]["COMBO"] / R[n]["costs"]["F_def"]} for n in NAMES])
cost.to_csv(OUT / "cost.csv", index=False, float_format="%.3f")
print("联邦计算量代理（轮数 × 深度 × 行采样 × 模型个数）：")
print(cost.round(2).to_string(index=False))

联邦计算量代理（轮数 × 深度 × 行采样 × 模型个数）：
             数据集  选定组合  原版联邦  优化版联邦  选定组合计算量  组合 / 原版
    santander_g3    M1 500.0  198.0    246.0     0.49
   mock_balanced  P+M6 500.0  528.0   2534.4     5.07
  mock_overlap20  P_M4 500.0  260.0    298.2     0.60
  mock_overlap50  P_M4 500.0  372.0    378.0     0.76
  mock_overlap80  P+M6 500.0  498.0   2390.4     4.78
        uci_bank    M1 500.0  136.0     10.0     0.02
    orange_churn    M1 500.0  228.0     39.0     0.08
   orange_upsell  P_M4 500.0  306.0    219.0     0.44
orange_appetency    M1 500.0  216.0     15.0     0.03
          fedads     P 500.0   56.0     56.0     0.11
      homecredit    M1 500.0  756.0    528.0     1.06
    santander_tx M1+M6 500.0  891.0   2217.6     4.44
      ieee_part0    M1 500.0  672.0     42.0     0.08
        coil2000    M1 500.0  177.0     24.0     0.05


## 读法

- **普遍有效**：
  - I2 + I3 在 11 / 14 个任务上显著，+0.005 ~ +0.048；
  - M6 bagging 在 7 / 14 上再显著提高 +0.004 ~ +0.010，没有一个变差，计算量约 4.8 倍。
- **联邦值不值得做取决于 I2**：原版参数下，联邦只在 7 / 14 上胜过同等调参的内地单边，优化后变成 10 / 14。FedAds、Orange 流失、Orange 偏好都是从「无增益」变为「有增益」。
- **只在个别任务上有增益**：
  - I3：FedAds +0.026；
  - M1x：FedAds +0.009；
  - M1：Santander 交易 +0.008，但在 5 个任务上变差；
  - M2 融合：Santander 交易 +0.001；
  - M2 蒸馏学生：mock 平衡 +0.007。
- **无增益**：M4 行采样、M5 匹配置信度加权 / 剔除。
- **标签 DP**：
  - 软标签在内地单边较强的任务上优于硬 RR，在香港特征主导的任务上更差；
  - ε ≤ 2 时，加噪后的联邦在多数任务上低于内地单边。
- **组合**：验证集选定的组合在 3 / 14 上优于优化版，在 2 / 14 上变差（Orange 流失、偏好，选了 M1）。
- **计算量**：含 M1 的组合，联邦部分只有原版的 2% ~ 8%（UCI、Orange 流失 / 偏好、COIL、IEEE）。精度不稳，但可作为省密文计算、少暴露树的选项。
- 全部为代理结论，待真实 SGB 复核（S5.M2）；开发证据，不代表汇丰效果。